In [1]:
import json
import os
import random
import time
import requests
import PyPDF2
from config.paths import POLICIES_DIR, DOWNLOAD_LINKS_PATH
from dotenv import dotenv_values
from common import logger

env = dotenv_values()


class PolicyExtractionPipeline:
    def __init__(self):
        """
        Initializes the PolicyExtractionPipeline with the given download directory and links JSON path.

        :param download_dir: Directory where files will be downloaded.
        :param links_json_path: Path to the JSON file containing the download links.
        """
        self.download_folder = POLICIES_DIR
        self.downloaded_files = set()
        self.links_dict = json.load(open(DOWNLOAD_LINKS_PATH, "r"))
        self.cookiesession1 = env["CMF_COOKIESESSION1"]
        self.phpsessid = env["CMF_PHPSESSID"]

    def download_files_from_json(self):
        """
        Downloads files from the links provided in the JSON file to the specified download directory.
        """
        if not os.path.exists(self.download_folder):
            os.makedirs(self.download_folder)

        if os.path.exists("downloaded.json"):
            with open("downloaded.json", "r") as downloaded_file:
                self.downloaded_files = set(json.load(downloaded_file))

        consecutive_failures = 0

        for code, link in self.links_dict.items():
            try:
                sleep_time = random.uniform(5, 10)
                time.sleep(sleep_time)
                headers = {
                    'User-Agent': 'Mozilla/5.0',
                    'Cookie': f'cookiesession1={self.cookiesession1}; PHPSESSID={self.phpsessid}'
                }
                response = requests.get(link, headers=headers)
                if response.status_code == 200:
                    file_name = os.path.join(
                        self.download_folder, code + '.pdf')
                    with open(file_name, 'wb') as f:
                        f.write(response.content)
                    logger.debug(f"Downloaded file for code {code}")

                    self.downloaded_files.add(code)
                    consecutive_failures = 0
                else:
                    logger.error(
                        f"Failed to download file for code {code}: Status code {response.status_code}")
                    consecutive_failures += 1
                    if consecutive_failures >= 5:
                        logger.error(
                            "Too many consecutive failures. Cancelling process.")
                        break

            except Exception as e:
                logger.error(f"Failed to download file for code {code}: {e}")
                consecutive_failures += 1
                if consecutive_failures >= 5:
                    logger.error(
                        "Too many consecutive failures. Cancelling process.")
                    break

        with open("downloaded.json", "w") as downloaded_file:
            json.dump(list(self.downloaded_files), downloaded_file)

    def check_missing_files(self):
        """
        Checks for any missing files that were not downloaded.

        :return: Set of missing codes.
        """
        downloaded_files_json_list = json.load(open("downloaded.json", "r"))
        self.downloaded_files = set(downloaded_files_json_list)
        all_codes = set(self.links_dict.keys())
        missing_codes = all_codes - self.downloaded_files
        return missing_codes

    def convert_doc_files(self):
        """
        Converts downloaded files to .doc format if they cannot be read as PDFs.
        """
        renamed_files = []
        failed_files = []

        for file_name in os.listdir(self.download_folder):
            file_path = os.path.join(self.download_folder, file_name)
            try:
                with open(file_path, 'rb') as file:
                    reader = PyPDF2.PdfReader(file)
            except Exception as e:
                logger.debug(f"Failed to open file {file_name} as PDF: {e}")
                new_file_path = os.path.join(
                    self.download_folder, os.path.splitext(file_name)[0] + ".doc")
                try:
                    os.rename(file_path, new_file_path)
                    renamed_files.append(new_file_path)
                    logger.debug(
                        f"Renamed file {file_name} to {new_file_path}")
                except Exception as e:
                    logger.debug(f"Failed to rename file {file_name}: {e}")
                    failed_files.append(file_path)

        logger.debug(f"Renamed files: {renamed_files}")
        logger.debug(f"Failed files: {failed_files}")

    def convert_rtf_files(self):
        """
        Converts files identified as RTF to have the .rtf extension.

        :return: List of renamed files.
        """
        renamed_files = []

        for file_name in os.listdir(self.download_folder):
            file_path = os.path.join(self.download_folder, file_name)
            logger.debug(f"Checking file {file_name}")
            try:
                with open(file_path, 'r') as file:
                    start_chars = file.read(10)
                    logger.debug(f"Start chars for {file_name}: {start_chars}")

                if "\\rtf" in start_chars and not file_name.endswith(".rtf"):
                    logger.debug(f"{file_name} is an RTF file")
                    new_file_path = os.path.join(
                        self.download_folder, os.path.splitext(file_name)[0] + ".rtf")
                    os.rename(file_path, new_file_path)
                    renamed_files.append(new_file_path)
                    logger.debug(
                        f"Renamed file {file_name} to {new_file_path}")

            except Exception as e:
                logger.debug(f"Failed to open file {file_name}: {e}")

        return renamed_files


In [ ]:
extraction_pipe = PolicyExtractionPipeline()
extraction_pipe.download_files_from_json()

In [ ]:
extraction_pipe.convert_doc_files()
extraction_pipe.convert_rtf_files()

In [1]:
from config.paths import x, POLICIES_DIR
from striprtf.striprtf import rtf_to_text
from common.logger import logger
from PyPDF2 import PdfReader
from tqdm import tqdm
import pandas as pd
import win32com.client
import datetime
import django
import json
import re
import os

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.environments.dev')
django.setup()


def transform_and_load_policies():
    """
    Fetches all the existing documents from the .csv file and stores them into the SQLITE database after
    cleaning and transforming the data.
    """
    #! Needs django initalized to work
    #! Only works on Windows since it uses win32com

    from apps.docs.models import Policy

    df = pd.read_csv(POLICIES_DATA_PATH)
    logger.info(f"Storing {df.shape[0]} policies into the database.")

    word = win32com.client.Dispatch("Word.Application")
    all_topics = set()

    for _, row in df.iterrows():
        data = {}
        data["topics"] = json.dumps(topics)
        code = row["code"]

        file = None
        if os.path.exists(os.path.join(POLICIES_DIR, f"{code}.pdf")):
            file = os.path.join(POLICIES_DIR, f"{code}.pdf")
        elif os.path.exists(os.path.join(POLICIES_DIR, f"{code}.doc")):
            file = os.path.join(POLICIES_DIR, f"{code}.doc")
        elif os.path.exists(os.path.join(POLICIES_DIR, f"{code}.rtf")):
            file = os.path.join(POLICIES_DIR, f"{code}.rtf")

        if not file:
            logger.error(f"File {code} not found.")
        else:
            with open(file, 'rb') as f:
                if file.endswith('.pdf'):
                    reader = PdfReader(f)
                    content = ''.join(page.extract_text()
                                      for page in reader.pages)
                    page_count = len(reader.pages)
                elif file.endswith('.doc'):
                    try:
                        wb = word.Documents.Open(file, False, False, True)
                        doc = word.ActiveDocument
                        content = doc.Range().Text
                        page_count = doc.ComputeStatistics(2)
                        wb.Close(SaveChanges=False)
                    except Exception as e:
                        logger.error(f"An error occurred: {e}")
                        continue
                elif file.endswith('.rtf'):
                    content = rtf_to_text(f.read())
                    page_count = None

                data["content"] = content
                data["page_count"] = page_count
                data["word_count"] = len(content.split())
                data["char_count"] = len(content)

        logger.info(f"Storing policy {code} into the database.")
        policy = Policy.objects.create(**data)
        policy.save()

    logger.info("Policies stored successfully.")
    word.Quit()

    logger.info("Updating policy topics.")
    updated = 0
    policies = Policy.objects.all()

    for policy in policies:
        topics = json.loads(policy.topics)
        topics = [topic.lower() for topic in topics]
        policy.topics = json.dumps(topics)
        policy.save()
        updated += 1
        logger.info(f"Policy {updated}/{len(policies)} updated.")

    logger.info(
        f"Topics updated successfully. {len(all_topics)} topics found.")
    logger.info(f"Topics: {all_topics}")

#   logger.info("Fixing last policy details.")
#   for policy in tqdm(policies):
##       policy.title = policy.title.upper()
#       policy.save()

    logger.info("Last policy fixes applied successfully.")


def load_to_chroma():
    from apps.docs.models import Policy
    from apps.ai.core.chroma import ChromaManager

    chroma_manager = ChromaManager()
    chroma_manager.reset_all_collections()
    policies = Policy.objects.all()

    for policy in tqdm(policies, desc="Creating embeddings for policy titles..."):
        embedding_str = f"{policy.title}"
        metadata = {
            "title": policy.title,
            "type": policy.type,
            "deposit_date": policy.deposit_date.strftime("%Y-%m-%d"),
            "linked_policies": json.dumps(policy.linked_policies),
            "insurance_company": policy.insurance_company,
            "prohibition_resolution": policy.prohibition_resolution,
            "authorization_resolution": policy.authorization_resolution,
            "is_prohibited": policy.is_prohibited,
            "topics": policy.topics,
            "char_count": policy.char_count,
            "word_count": policy.word_count,
            "page_count": policy.page_count,
        }

        chroma_manager.add_entry(
            collection_name="documents",
            pol_id=policy.code,
            str_embedding=embedding_str,
            metadata=metadata
        )

    logger.info("All policies inserted into the collection.")


c:\Users\abeng\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
c:\Users\abeng\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
